# Pull Data

Interactive front-end for the collection pipeline in `src/`. Each cell below runs one of
the existing scripts by simulating its CLI invocation (`sys.argv` + `parse_args()`), so this
notebook stays in sync with the scripts automatically -- no logic is duplicated here.

Run cells top to bottom. The "pull your own games" section is the one you'll use most often;
the full-pool refresh section is for occasionally expanding the Diamond/Master/GM/Challenger
training population and can take a long time (Riot's dev key rate limit is ~1 request/1.2s).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))
os.chdir('..')  # so relative data/ paths in the scripts resolve correctly

## 1. Pull your own recent games

Paginates through match-v5 for the given Riot ID(s) until it runs out of history (or hits
`--max-games`), and appends only new match IDs/details -- safe to re-run any time.
Requires a valid `RIOT_API_KEY` in `.env` (keys expire roughly every 24h).

In [ ]:
import fetch_recent_self

RIOT_ID = 'river3#na1'
MAX_GAMES = 0  # 0 = pull all available history

sys.argv = ['fetch_recent_self.py', '--riot-id', RIOT_ID, '--max-games', str(MAX_GAMES)]
fetch_recent_self.run(fetch_recent_self.parse_args())

## 2. Rebuild the feature table

Re-flattens `data/raw/matches.ndjson` (now including whatever you just pulled) into
`data/processed/participant_features.parquet`. Cheap to re-run -- it rescans everything,
but that's a few seconds even at tens of thousands of rows.

In [ ]:
import build_features

sys.argv = ['build_features.py']
build_features.run(build_features.parse_args())

## 3. (Optional) Refresh the full ranked pool

Only needed occasionally -- to add more Diamond/Master/GM/Challenger players to the training
population, not for your own games. Each tier writes to its own file (see README for why:
apex tiers don't carry a `tier` field per row, so `build_features.py` infers it from the
filename). This can take a while for the Diamond page fetch and the subsequent match ID /
match detail stages -- consider running these from a terminal instead if kicking off a large
refresh, so it isn't tied to this notebook's kernel.

In [ ]:
import fetch_league_pool

# Uncomment the tier(s) you want to refresh.

# sys.argv = ['fetch_league_pool.py', '--tier', 'DIAMOND', '--division', 'II']
# fetch_league_pool.fetch_player_pool(fetch_league_pool.parse_args())

# sys.argv = ['fetch_league_pool.py', '--tier', 'MASTER', '--output-path', 'data/raw/league_entries_master.json']
# fetch_league_pool.fetch_player_pool(fetch_league_pool.parse_args())

# sys.argv = ['fetch_league_pool.py', '--tier', 'GRANDMASTER', '--output-path', 'data/raw/league_entries_grandmaster.json']
# fetch_league_pool.fetch_player_pool(fetch_league_pool.parse_args())

# sys.argv = ['fetch_league_pool.py', '--tier', 'CHALLENGER', '--output-path', 'data/raw/league_entries_challenger.json']
# fetch_league_pool.fetch_player_pool(fetch_league_pool.parse_args())

In [ ]:
import fetch_match_ids
import fetch_match_details

# After refreshing the pool above, pull match IDs and details for it (resumable -- rerun with
# --resume if it gets interrupted). This scans the ENTIRE pool, so it's the slowest step.

# sys.argv = ['fetch_match_ids.py', '--resume']
# fetch_match_ids.run(fetch_match_ids.parse_args())

# sys.argv = ['fetch_match_details.py', '--resume']
# fetch_match_details.run(fetch_match_details.parse_args())